# Análise Exploratória — Oportunidades em meio ao Caos

Esta etapa avalia a qualidade dos dados depois do pré-processamento e investiga padrões que podem ajudar na futura classificação de regimes de estresse e na análise de resiliência dos setores.

A pergunta principal do projeto continua sendo:

> **Dado um cenário de estresse socioeconômico, quais setores historicamente apresentaram maior resiliência e melhor relação entre retorno e risco?**

A EDA não tem como objetivo provar causalidade nem treinar o modelo final. O foco é entender a base, suas limitações e quais hipóteses fazem sentido levar para a próxima etapa.

## 1. Carregamento e configuração

O notebook procura primeiro os arquivos em `data/curated`. Se não encontrar, tenta ler o bucket `curated` do MinIO. Assim, o notebook funciona tanto localmente quanto com o armazenamento do projeto.

In [ ]:
from pathlib import Path
from io import BytesIO
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Descobre a raiz do projeto quando o notebook roda dentro de notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "data" / "curated").exists():
    BASE = CWD
elif (CWD.parent / "data" / "curated").exists():
    BASE = CWD.parent
else:
    BASE = CWD

CURATED = BASE / "data" / "curated"
load_dotenv(BASE / ".env")

print("Diretório do projeto:", BASE)
print("Curated local:", CURATED)

In [ ]:
def read_curated(filename: str) -> pd.DataFrame:
    local = CURATED / filename
    if local.exists():
        print(f"LOCAL  -> {local}")
        return pd.read_csv(local)

    # Fallback opcional para MinIO.
    import boto3

    endpoint = os.getenv("MINIO_ENDPOINT", "localhost:9000")
    secure = os.getenv("MINIO_SECURE", "false").lower() in {"1", "true", "yes"}
    access = os.getenv("MINIO_ROOT_USER") or os.getenv("MINIO_ACCESS_KEY")
    secret = os.getenv("MINIO_ROOT_PASSWORD") or os.getenv("MINIO_SECRET_KEY")
    bucket = os.getenv("MINIO_BUCKET_CURATED", "curated")

    if not access or not secret:
        raise FileNotFoundError(
            f"{filename} não encontrado localmente e credenciais do MinIO não estão configuradas."
        )

    s3 = boto3.client(
        "s3",
        endpoint_url=f"http{'s' if secure else ''}://{endpoint}",
        aws_access_key_id=access,
        aws_secret_access_key=secret,
        region_name="us-east-1",
    )
    obj = s3.get_object(Bucket=bucket, Key=filename)
    print(f"MINIO  -> s3://{bucket}/{filename}")
    return pd.read_csv(BytesIO(obj["Body"].read()))

ref = read_curated("dataset_monthly_reference.csv")
mvp = read_curated("dataset_mvp.csv")
complete = read_curated("dataset_mvp_complete.csv")

for df in (ref, mvp, complete):
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)

print("\nShapes:")
print("reference:", ref.shape)
print("mvp:      ", mvp.shape)
print("complete: ", complete.shape)

## 2. Entendimento dos três datasets

- **`dataset_monthly_reference.csv`**: dados alinhados ao mês de referência. É o principal dataset para gráficos históricos e EDA.
- **`dataset_mvp.csv`**: contém os lags aplicados no pré-processamento para reduzir *look-ahead bias*. É a referência para avaliar futuras features do modelo.
- **`dataset_mvp_complete.csv`**: mantém somente meses em que todas as variáveis principais estão preenchidas. É útil para algumas análises conjuntas, mas não deve limitar toda a EDA.

In [ ]:
def dataset_summary(name, df):
    return {
        "dataset": name,
        "linhas": len(df),
        "colunas": df.shape[1],
        "inicio": df["date"].min(),
        "fim": df["date"].max(),
        "duplicadas_data": int(df["date"].duplicated().sum()),
        "missing_total": int(df.isna().sum().sum()),
    }

summary_datasets = pd.DataFrame([
    dataset_summary("monthly_reference", ref),
    dataset_summary("mvp", mvp),
    dataset_summary("mvp_complete", complete),
])
display(summary_datasets)

### Atenção ao último mês

O pré-processamento deve trabalhar apenas com meses completos. Como proteção adicional, a EDA identifica o último mês calendário já encerrado e evita usar um mês ainda em andamento nas comparações temporais.

In [ ]:
today = pd.Timestamp.today().normalize()
last_closed_month = today.to_period("M").start_time - pd.Timedelta(days=1)
last_closed_month = last_closed_month.to_period("M").to_timestamp("M")

print("Hoje:", today.date())
print("Último mês calendário encerrado:", last_closed_month.date())

for name, df in [("reference", ref), ("mvp", mvp), ("complete", complete)]:
    future_rows = df[df["date"] > last_closed_month]
    print(f"{name}: {len(future_rows)} linha(s) após o último mês encerrado")

# Bases de análise sem mês calendário incompleto.
ref_eda = ref[ref["date"] <= last_closed_month].copy()
mvp_eda = mvp[mvp["date"] <= last_closed_month].copy()
complete_eda = complete[complete["date"] <= last_closed_month].copy()

## 3. Qualidade dos dados

Aqui verificamos tipos, missing values, duplicidades e continuidade mensal. Missing não será preenchido automaticamente: primeiro precisamos entender por que existe.

In [ ]:
quality = pd.DataFrame({
    "coluna": ref_eda.columns,
    "tipo": [str(ref_eda[c].dtype) for c in ref_eda.columns],
    "non_null": [int(ref_eda[c].notna().sum()) for c in ref_eda.columns],
    "missing": [int(ref_eda[c].isna().sum()) for c in ref_eda.columns],
    "missing_pct": [round(ref_eda[c].isna().mean() * 100, 2) for c in ref_eda.columns],
})
display(quality.sort_values("missing_pct", ascending=False))

print("Linhas duplicadas completas:", ref_eda.duplicated().sum())
print("Datas duplicadas:", ref_eda["date"].duplicated().sum())

In [ ]:
# Verifica meses ausentes entre início e fim da base.
periods = ref_eda["date"].dt.to_period("M")
expected = pd.period_range(periods.min(), periods.max(), freq="M")
missing_months = expected.difference(pd.Index(periods.unique()))

print("Meses esperados:", len(expected))
print("Meses presentes:", periods.nunique())
print("Meses ausentes:", len(missing_months))
if len(missing_months):
    display(pd.DataFrame({"mes_ausente": missing_months.astype(str)}))

## 4. Cobertura histórica por variável

Essa análise ajuda a não perder histórico desnecessariamente. Uma variável pode ter poucos anos de cobertura sem impedir análises que não dependem dela.

In [ ]:
def coverage_table(df):
    rows = []
    for col in df.columns:
        if col == "date":
            continue
        valid = df.loc[df[col].notna(), ["date", col]]
        rows.append({
            "variavel": col,
            "inicio": valid["date"].min() if len(valid) else pd.NaT,
            "fim": valid["date"].max() if len(valid) else pd.NaT,
            "observacoes": int(df[col].notna().sum()),
            "missing": int(df[col].isna().sum()),
            "cobertura_pct": round(df[col].notna().mean() * 100, 2),
        })
    return pd.DataFrame(rows).sort_values(["cobertura_pct", "variavel"])

coverage = coverage_table(ref_eda)
display(coverage)

In [ ]:
# Compara a janela macroeconômica com a janela em que todos os setores estão disponíveis.
macro_cols = [
    "ibc_br", "selic", "usd_brl", "ipca_month", "ipca_12m", "pib_index", "unemployment"
]
sector_cols = ["ibovespa", "ifnc", "icon", "iee"]

macro_available = mvp_eda.dropna(subset=[c for c in macro_cols if c in mvp_eda.columns])
sector_available = ref_eda.dropna(subset=[c for c in sector_cols if c in ref_eda.columns])

comparison_history = pd.DataFrame([
    {
        "conjunto": "Macroeconômico completo",
        "linhas": len(macro_available),
        "inicio": macro_available["date"].min(),
        "fim": macro_available["date"].max(),
    },
    {
        "conjunto": "Todos os índices B3",
        "linhas": len(sector_available),
        "inicio": sector_available["date"].min(),
        "fim": sector_available["date"].max(),
    },
    {
        "conjunto": "MVP complete",
        "linhas": len(complete_eda),
        "inicio": complete_eda["date"].min(),
        "fim": complete_eda["date"].max(),
    },
])
display(comparison_history)

## 5. Estrutura das variáveis

As variáveis são separadas em dois papéis:

- **Regime econômico**: candidatas futuras para classificar períodos de normalidade ou estresse.
- **Mercado/resiliência**: usadas para medir como Ibovespa, IFNC, ICON e IEE se comportaram.

In [ ]:
regime_features = [c for c in [
    "ibc_br", "ibc_br_change", "selic", "selic_change",
    "usd_brl", "usd_brl_return", "usd_brl_volatility",
    "ipca_month", "ipca_12m", "pib_index", "pib_change_3m",
    "unemployment", "unemployment_change"
] if c in mvp_eda.columns]

market_features = [c for c in ref_eda.columns if any(
    c.startswith(prefix) for prefix in ["ibovespa", "ifnc", "icon", "iee"]
)]

print("Features de regime:")
print(regime_features)
print("\nVariáveis de mercado/resiliência:")
print(market_features)

## 6. Estatística descritiva

Além de média e mediana, usamos desvio padrão, quartis, assimetria e curtose. Assimetria ajuda a identificar distribuições inclinadas; curtose elevada pode indicar caudas pesadas e maior presença de valores extremos.

In [ ]:
main_numeric = [c for c in [
    "ibc_br_change", "selic", "selic_change", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment", "unemployment_change",
    "ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

stats = pd.DataFrame(index=main_numeric)
stats["n"] = mvp_eda[main_numeric].count()
stats["media"] = mvp_eda[main_numeric].mean()
stats["mediana"] = mvp_eda[main_numeric].median()
stats["desvio_padrao"] = mvp_eda[main_numeric].std()
stats["min"] = mvp_eda[main_numeric].min()
stats["q25"] = mvp_eda[main_numeric].quantile(.25)
stats["q75"] = mvp_eda[main_numeric].quantile(.75)
stats["max"] = mvp_eda[main_numeric].max()
stats["assimetria"] = mvp_eda[main_numeric].skew()
stats["curtose"] = mvp_eda[main_numeric].kurt()
display(stats)

## 7. Distribuição das principais variáveis

Histogramas mostram concentração e formato da distribuição. Boxplots ajudam a localizar dispersão e valores extremos. Os extremos não são removidos: eles podem representar justamente momentos de estresse.

In [ ]:
dist_cols = [c for c in [
    "ibc_br_change", "selic", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment",
    "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

for col in dist_cols:
    s = mvp_eda[col].dropna()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(s, bins=min(20, max(8, int(np.sqrt(len(s))))), edgecolor="black", alpha=.75)
    ax.axvline(s.mean(), linestyle="--", label=f"Média {s.mean():.3f}")
    ax.axvline(s.median(), linestyle=":", label=f"Mediana {s.median():.3f}")
    ax.set_title(f"Distribuição — {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequência")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
return_cols = [c for c in ["ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"] if c in ref_eda.columns]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([ref_eda[c].dropna() for c in return_cols], tick_labels=return_cols, showfliers=True)
ax.axhline(0, linewidth=1)
ax.set_title("Distribuição dos retornos mensais — índices B3")
ax.set_ylabel("Retorno mensal")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 8. Evolução temporal

Gráficos de linha ajudam a observar tendência, mudanças de nível e possíveis quebras. Variáveis com escalas muito diferentes são mostradas separadamente.

In [ ]:
time_series_cols = [c for c in [
    "ibc_br", "selic", "usd_brl", "usd_brl_volatility", "ipca_12m", "pib_index", "unemployment"
] if c in ref_eda.columns]

for col in time_series_cols:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(ref_eda["date"], ref_eda[col])
    ax.set_title(f"Evolução temporal — {col}")
    ax.set_xlabel("Data")
    ax.set_ylabel(col)
    ax.grid(alpha=.25)
    plt.tight_layout()
    plt.show()

### Índices normalizados para base 100

Os índices possuem níveis diferentes. Para comparar apenas a trajetória relativa, cada série é normalizada para começar em 100. Os valores originais não são alterados.

In [ ]:
indices = [c for c in ["ibovespa", "ifnc", "icon", "iee"] if c in ref_eda.columns]
base_idx = ref_eda[["date"] + indices].dropna(subset=indices).copy()

normalized = base_idx[["date"]].copy()
for col in indices:
    normalized[col] = base_idx[col] / base_idx[col].iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for col in indices:
    ax.plot(normalized["date"], normalized[col], label=col.upper())
ax.set_title("Índices B3 normalizados — base 100")
ax.set_xlabel("Data")
ax.set_ylabel("Base 100")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 9. Retornos dos setores

Retorno mensal é a variação percentual do índice em relação ao mês anterior. Aqui comparamos retorno médio, dispersão, melhores e piores meses e frequência de meses positivos/negativos.

In [ ]:
rows = []
for index_name in ["ibovespa", "ifnc", "icon", "iee"]:
    col = f"{index_name}_return_1m"
    if col not in ref_eda:
        continue
    tmp = ref_eda[["date", col]].dropna()
    best = tmp.loc[tmp[col].idxmax()]
    worst = tmp.loc[tmp[col].idxmin()]
    rows.append({
        "indice": index_name.upper(),
        "n": len(tmp),
        "retorno_medio": tmp[col].mean(),
        "mediana": tmp[col].median(),
        "desvio_padrao": tmp[col].std(),
        "melhor_mes": best[col],
        "data_melhor": best["date"],
        "pior_mes": worst[col],
        "data_pior": worst["date"],
        "meses_positivos_pct": (tmp[col] > 0).mean() * 100,
        "meses_negativos_pct": (tmp[col] < 0).mean() * 100,
    })

sector_returns = pd.DataFrame(rows)
display(sector_returns)

## 10. Volatilidade

No pré-processamento, a volatilidade dos índices é calculada pelo desvio padrão dos últimos três retornos mensais e anualizada por `sqrt(12)`. Para o dólar, a volatilidade parte dos retornos diários do mês e é escalada por `sqrt(21)`.

In [ ]:
vol_cols = [c for c in [
    "usd_brl_volatility", "ibovespa_volatility_3m_ann", "ifnc_volatility_3m_ann",
    "icon_volatility_3m_ann", "iee_volatility_3m_ann"
] if c in ref_eda.columns]

fig, ax = plt.subplots(figsize=(12, 5))
for col in vol_cols:
    ax.plot(ref_eda["date"], ref_eda[col], label=col)
ax.set_title("Volatilidade ao longo do tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Volatilidade")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

display(ref_eda[vol_cols].describe().T[["mean", "50%", "std", "max"]])

## 11. Drawdown

Drawdown mede quanto o índice está abaixo do maior valor observado até aquele momento. É uma medida importante para resiliência porque mostra profundidade das perdas, e não apenas retorno médio.

In [ ]:
dd_cols = [c for c in ["ibovespa_drawdown", "ifnc_drawdown", "icon_drawdown", "iee_drawdown"] if c in ref_eda.columns]

fig, ax = plt.subplots(figsize=(12, 5))
for col in dd_cols:
    ax.plot(ref_eda["date"], ref_eda[col], label=col)
ax.set_title("Drawdown dos índices B3")
ax.set_xlabel("Data")
ax.set_ylabel("Drawdown")
ax.axhline(0, linewidth=1)
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

worst_dd = []
for col in dd_cols:
    tmp = ref_eda[["date", col]].dropna()
    row = tmp.loc[tmp[col].idxmin()]
    worst_dd.append({"indice": col.replace("_drawdown", "").upper(), "pior_drawdown": row[col], "data": row["date"]})
display(pd.DataFrame(worst_dd).sort_values("pior_drawdown"))

## 12. Períodos econômicos relevantes

Os intervalos abaixo são usados como segmentos descritivos, não como rótulos definitivos de `stress`. O objetivo é comparar se os setores reagiram de maneira diferente em contextos conhecidos.

In [ ]:
periods = {
    "Recessão 2015-2016": ("2015-01-01", "2016-12-31"),
    "Pré-pandemia 2018-2019": ("2018-01-01", "2019-12-31"),
    "Pandemia 2020": ("2020-01-01", "2020-12-31"),
    "Inflação/juros 2021-2022": ("2021-01-01", "2022-12-31"),
    "2023 em diante": ("2023-01-01", str(last_closed_month.date())),
}

def period_metrics(df, period_name, start, end, index_name):
    ret_col = f"{index_name}_return_1m"
    dd_col = f"{index_name}_drawdown"
    tmp = df.loc[df["date"].between(pd.Timestamp(start), pd.Timestamp(end)), ["date", ret_col, dd_col]].dropna(subset=[ret_col])
    if tmp.empty:
        return None
    compounded = (1 + tmp[ret_col]).prod() - 1
    return {
        "periodo": period_name,
        "indice": index_name.upper(),
        "meses": len(tmp),
        "retorno_acumulado": compounded,
        "retorno_medio_mensal": tmp[ret_col].mean(),
        "volatilidade_mensal": tmp[ret_col].std(),
        "pior_drawdown_observado": tmp[dd_col].min(),
        "melhor_mes": tmp[ret_col].max(),
        "pior_mes": tmp[ret_col].min(),
    }

rows = []
for pname, (start, end) in periods.items():
    for idx_name in ["ibovespa", "ifnc", "icon", "iee"]:
        r = period_metrics(ref_eda, pname, start, end, idx_name)
        if r:
            rows.append(r)
period_table = pd.DataFrame(rows)
display(period_table)

## 13. Correlações

A correlação será usada apenas para levantar relações candidatas. **Correlação não implica causalidade.** Também mostramos Pearson e Spearman: Pearson destaca relações lineares; Spearman verifica se a ordem das variáveis se move de maneira monotônica, sendo menos sensível a alguns extremos.

In [ ]:
corr_cols = [c for c in [
    "ibc_br_change", "selic", "selic_change", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment", "unemployment_change",
    "ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

corr_data = mvp_eda[corr_cols]
pearson = corr_data.corr(method="pearson")
spearman = corr_data.corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(pearson, vmin=-1, vmax=1, cmap="coolwarm")
n = len(pearson.columns)
ax.set_xticks(range(n), pearson.columns, rotation=90)
ax.set_yticks(range(n), pearson.columns)
fig.colorbar(im, ax=ax, label="Correlação de Pearson")
ax.set_title("Matriz de correlação — variáveis selecionadas")
plt.tight_layout()
plt.show()

In [ ]:
def strongest_pairs(corr, top_n=12):
    pairs = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append((cols[i], cols[j], corr.iloc[i, j]))
    out = pd.DataFrame(pairs, columns=["var_1", "var_2", "correlacao"])
    out["abs_corr"] = out["correlacao"].abs()
    return out.sort_values("abs_corr", ascending=False).head(top_n)

print("Relações lineares mais fortes (Pearson):")
display(strongest_pairs(pearson))

print("Relações monotônicas mais fortes (Spearman):")
display(strongest_pairs(spearman))

## 14. Relações entre macroeconomia e setores

Algumas relações são analisadas diretamente porque fazem sentido para o problema. Os gráficos são exploratórios e não devem ser lidos como evidência de causa e efeito.

In [ ]:
relations = [
    ("selic", "ifnc_return_1m"),
    ("selic", "icon_return_1m"),
    ("selic", "iee_return_1m"),
    ("ipca_12m", "icon_return_1m"),
    ("usd_brl_return", "ifnc_return_1m"),
    ("usd_brl_return", "icon_return_1m"),
    ("usd_brl_return", "iee_return_1m"),
    ("ibc_br_change", "icon_return_1m"),
    ("unemployment", "icon_return_1m"),
]

relation_rows = []
for x, y in relations:
    if x not in mvp_eda or y not in mvp_eda:
        continue
    tmp = mvp_eda[[x, y]].dropna()
    if len(tmp) < 10:
        continue
    p = tmp[x].corr(tmp[y], method="pearson")
    s = tmp[x].corr(tmp[y], method="spearman")
    relation_rows.append({"x": x, "y": y, "n": len(tmp), "pearson": p, "spearman": s})

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(tmp[x], tmp[y], alpha=.65)
    if tmp[x].nunique() > 1:
        coef = np.polyfit(tmp[x], tmp[y], 1)
        xx = np.linspace(tmp[x].min(), tmp[x].max(), 100)
        ax.plot(xx, coef[0] * xx + coef[1], linestyle="--")
    ax.set_title(f"{x} x {y} | Pearson={p:.2f}")
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()

display(pd.DataFrame(relation_rows).sort_values("pearson", key=lambda s: s.abs(), ascending=False))

## 15. Outliers e anomalias

Usamos extremos e a regra do IQR apenas para **identificar** pontos que merecem investigação. Nenhuma linha é removida automaticamente.

In [ ]:
outlier_cols = [c for c in [
    "ibc_br_change", "usd_brl_return", "usd_brl_volatility",
    "ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

extreme_rows = []
for col in outlier_cols:
    tmp = mvp_eda[["date", col]].dropna()
    for _, r in tmp.nsmallest(5, col).iterrows():
        extreme_rows.append({"variavel": col, "tipo": "5 menores", "date": r["date"], "valor": r[col]})
    for _, r in tmp.nlargest(5, col).iterrows():
        extreme_rows.append({"variavel": col, "tipo": "5 maiores", "date": r["date"], "valor": r[col]})

extremes = pd.DataFrame(extreme_rows).sort_values(["variavel", "valor"])
display(extremes)

In [ ]:
iqr_rows = []
for col in outlier_cols:
    s = mvp_eda[col].dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()
    iqr_rows.append({
        "variavel": col,
        "limite_inferior": lower,
        "limite_superior": upper,
        "pontos_fora_IQR": int(count),
        "pct": round(count / len(s) * 100, 2),
    })
display(pd.DataFrame(iqr_rows))

## 16. Sazonalidade

Como a base é mensal, verificamos se os retornos apresentam diferenças recorrentes por mês do ano. Uma diferença visual não é suficiente para afirmar que existe sazonalidade estável.

In [ ]:
season = ref_eda[["date"] + return_cols].copy()
season["mes"] = season["date"].dt.month

seasonal_mean = season.groupby("mes")[return_cols].mean()
seasonal_median = season.groupby("mes")[return_cols].median()
seasonal_count = season.groupby("mes")[return_cols].count()

print("Retorno médio por mês do ano:")
display(seasonal_mean)
print("Quantidade de observações por mês:")
display(seasonal_count)

fig, ax = plt.subplots(figsize=(10, 5))
for col in return_cols:
    ax.plot(seasonal_mean.index, seasonal_mean[col], marker="o", label=col)
ax.axhline(0, linewidth=1)
ax.set_xticks(range(1, 13))
ax.set_title("Retorno médio por mês do ano")
ax.set_xlabel("Mês")
ax.set_ylabel("Retorno médio")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 17. Comparação por subperíodos

Médias gerais podem esconder mudanças de comportamento. Por isso comparamos alguns intervalos e verificamos se retorno e risco mudam entre contextos.

In [ ]:
subperiod_summary = period_table.pivot(index="periodo", columns="indice", values="retorno_acumulado")
display(subperiod_summary)

risk_summary = period_table.pivot(index="periodo", columns="indice", values="volatilidade_mensal")
display(risk_summary)

## 18. Redundância entre possíveis features

Features muito correlacionadas podem carregar informação parecida. Aqui apenas sinalizamos pares com correlação absoluta alta; a decisão de manter ou remover uma feature será feita na etapa de modelagem.

In [ ]:
regime_corr = mvp_eda[regime_features].corr()
high_corr = strongest_pairs(regime_corr, top_n=30)
high_corr = high_corr[high_corr["abs_corr"] >= 0.70]

if high_corr.empty:
    print("Nenhum par de features de regime com |correlação| >= 0.70.")
else:
    display(high_corr)

## 19. Volume disponível para Machine Learning

O número de linhas não é a única questão. Também importa quantas features serão usadas, quanto histórico cada uma possui e, futuramente, quantos meses serão rotulados como `stress` e `normal`.

Nesta etapa ainda não existe o target `stress`, então não é possível avaliar o balanceamento das classes.

In [ ]:
initial_feature_set = [c for c in [
    "ibc_br_change", "selic", "selic_change", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment", "unemployment_change"
] if c in mvp_eda.columns]

regime_ready = mvp_eda[["date"] + initial_feature_set].dropna()
all_market_ready = ref_eda[["date"] + sector_cols].dropna()

volume = pd.DataFrame([
    {
        "dataset": "MVP completo (todas core)",
        "linhas": len(complete_eda),
        "features/series": 11,
        "inicio": complete_eda["date"].min(),
        "fim": complete_eda["date"].max(),
    },
    {
        "dataset": "Features iniciais de regime",
        "linhas": len(regime_ready),
        "features/series": len(initial_feature_set),
        "inicio": regime_ready["date"].min(),
        "fim": regime_ready["date"].max(),
    },
    {
        "dataset": "Todos os índices B3",
        "linhas": len(all_market_ready),
        "features/series": len(sector_cols),
        "inicio": all_market_ready["date"].min(),
        "fim": all_market_ready["date"].max(),
    },
])
display(volume)

print("Features candidatas ao primeiro modelo:")
print(initial_feature_set)
print("\nObservações completas nessas features:", len(regime_ready))
print("Target 'stress' existe no dataset?", "stress" in mvp_eda.columns)

## 20. Dataset de regime x dataset de resiliência

A EDA permite avaliar uma separação importante:

- **Dataset de regime**: features macroeconômicas e socioeconômicas para classificar `stress/normal`.
- **Dataset de resiliência**: índices B3 e métricas de retorno, volatilidade e drawdown para comparar os setores.

Essa divisão evita perder histórico macroeconômico só porque um índice setorial possui menos dados e também deixa mais claro o papel de cada variável.

In [ ]:
regime_cols = ["date"] + initial_feature_set
resilience_cols = ["date"] + [c for c in ref_eda.columns if any(c.startswith(p) for p in ["ibovespa", "ifnc", "icon", "iee"])]

regime_dataset_preview = mvp_eda[regime_cols].dropna().copy()
resilience_dataset_preview = ref_eda[resilience_cols].dropna(subset=sector_cols).copy()

print("Dataset de regime (preview):", regime_dataset_preview.shape)
print(regime_dataset_preview["date"].min(), "->", regime_dataset_preview["date"].max())
print("Dataset de resiliência (preview):", resilience_dataset_preview.shape)
print(resilience_dataset_preview["date"].min(), "->", resilience_dataset_preview["date"].max())

## 21. Resumo automático dos principais achados

O bloco abaixo não substitui a interpretação. Ele resume alguns fatos objetivos para facilitar a documentação da entrega.

In [ ]:
# Cobertura mais limitante entre as principais séries.
core_for_coverage = [c for c in macro_cols + sector_cols if c in coverage["variavel"].values]
core_cov = coverage[coverage["variavel"].isin(core_for_coverage)].sort_values("observacoes")
limiting = core_cov.iloc[0]

# Setor com menor drawdown observado (mais negativo = pior perda).
dd_df = pd.DataFrame(worst_dd)
worst_sector = dd_df.loc[dd_df["pior_drawdown"].idxmin()]
best_sector = dd_df.loc[dd_df["pior_drawdown"].idxmax()]

print(f"Período da EDA: {ref_eda['date'].min().date()} a {ref_eda['date'].max().date()}")
print(f"Variável core com menor cobertura: {limiting['variavel']} ({int(limiting['observacoes'])} observações)")
print(f"Observações completas nas features iniciais de regime: {len(regime_ready)}")
print(f"Observações com todos os índices B3: {len(all_market_ready)}")
print(f"Maior queda observada entre os drawdowns: {worst_sector['indice']} = {worst_sector['pior_drawdown']:.2%}")
print(f"Menor pior-drawdown entre os índices comparados: {best_sector['indice']} = {best_sector['pior_drawdown']:.2%}")

## 22. Hipóteses para a próxima etapa

As hipóteses abaixo devem ser confirmadas ou descartadas usando os resultados apresentados nas células anteriores:

1. A volatilidade cambial pode aumentar em períodos em que os índices também apresentam maior risco.
2. IFNC, ICON e IEE podem responder de formas diferentes a mudanças na Selic.
3. ICON pode apresentar maior sensibilidade a deterioração de atividade econômica e mercado de trabalho.
4. A combinação de inflação alta, juros altos, câmbio e desaceleração pode ser mais útil para identificar regimes do que uma única variável isolada.
5. Retorno, volatilidade e drawdown devem ser analisados em conjunto antes de classificar um setor como resiliente.

Estas são hipóteses exploratórias, não conclusões causais.

## 23. Limitações identificadas

Pontos que devem ser considerados na próxima fase:

- frequência mensal reduz o número de observações;
- observações de séries temporais não são totalmente independentes;
- diferentes séries possuem coberturas históricas diferentes;
- PIB é trimestral e precisa de tratamento temporal;
- lags usados no MVP são aproximações conservadoras das datas reais de divulgação;
- são analisados apenas três índices setoriais na primeira versão;
- o target `stress` ainda precisa ser definido e documentado;
- modelos complexos podem sofrer overfitting com esse volume de dados.

A futura validação deve respeitar a ordem temporal, evitando um `train_test_split` aleatório comum.

## 24. Próximos passos

Depois desta EDA, a sequência sugerida é:

1. revisar os insights e anomalias encontradas;
2. definir formalmente o critério de `stress = 0/1`;
3. verificar a quantidade de meses em cada classe;
4. fechar o conjunto inicial de features;
5. criar um baseline;
6. testar Logistic Regression;
7. comparar com Random Forest controlando a complexidade;
8. usar validação temporal;
9. aplicar os regimes identificados à análise de IFNC, ICON e IEE;
10. construir uma medida/ranking de resiliência.